# Pipeline length estimatesThin wrapper around `route_lengths.py`. All of the logic lives in the module —edit that, not this notebook, so the scheduled run and the interactive run stayidentical.Prerequisites:- `pip install -r requirements.txt`- a boundary layer: `python prepare_boundaries.py` (~5 s), or the prepped  `.gpkg` from the *Automation inputs* folder on the work Drive- the `gws` work profile set up (`~/.config/gws-gem`) — reads are read-only

In [ ]:
import importlib
import pandas as pd

import route_lengths
importlib.reload(route_lengths)   # pick up module edits without restarting

pd.set_option('display.width', 200)

## ComputeReads the tracker tabs, the route geometry and the boundary layer, then measureseverything. Takes about a minute, dominated by reading ~6,400 route files.

In [ ]:
by_pipeline, by_country, diagnostics = route_lengths.compute()

## ReconciliationCheck this before doing anything with the results. `problems` is what wouldblock an automated write.

In [ ]:
lines, problems = route_lengths.reconciliation_report(by_pipeline, by_country, diagnostics)
print('\n'.join(lines))
print()
print('PROBLEMS:', problems or 'none')

## Results

In [ ]:
by_pipeline.head(10)

In [ ]:
by_country.head(10)

### Where the clipped total disagrees with the measured length`unattributed_km` is route length outside every boundary polygon — should be ~0.`overlap_km` is length counted twice because a named joint area overlaps aclaimant's own polygon, which is expected and is not an error.

In [ ]:
diagnostics.query('unattributed_km > 0.01 or overlap_km > 0.01') \
           .sort_values('overlap_km', ascending=False) \
           .head(20)

## Export

CSVs for review. Writing to the sheet is `sheet_writer.py`'s job — run it
from the cell below, or `python route_lengths.py --write` from a shell.


In [ ]:
out_dir = '/tmp/route-lengths'

import pathlib
pathlib.Path(out_dir).mkdir(parents=True, exist_ok=True)
by_pipeline.to_csv(f'{out_dir}/length-estimates-by-pipeline.csv', index=False)
by_country.to_csv(f'{out_dir}/country-ratios-by-pipeline.csv', index=False)
diagnostics.to_csv(f'{out_dir}/length-diagnostics.csv', index=False)
print('wrote', out_dir)

In [ ]:
import sheet_writer
importlib.reload(sheet_writer)

# Plan first -- this only reads.
plan = sheet_writer.plan(by_pipeline, by_country)
print(sheet_writer.describe(plan))


In [ ]:
# Uncomment to write. Refuses if the reconciliation report found problems.
# assert not problems, problems
# sheet_writer.write_all(by_pipeline, by_country, plan_dict=plan)
# print(sheet_writer.verify(plan) or 'verified')
